# Tarea C2 — Prototipo CNN 2D con Datos Dummy
**ECG Rhythm Analyzer | Gabriela — Modelo CNN**  
**Rama:** `feature/model-training`

## Objetivo
Construir el pipeline completo de entrenamiento usando imágenes **aleatorias** (224×224 píxeles).  
El modelo no aprenderá nada útil con datos falsos, pero si todo **corre sin errores**, el pipeline está listo para los datos reales de Gloria (Semana 3).

## Checklist de éxito
- [ ] 200 imágenes dummy creadas en la estructura correcta de carpetas
- [ ] EfficientNet-B0 cargado y adaptado (salida binaria)
- [ ] Data loader leyendo imágenes con normalización ImageNet
- [ ] Una epoch de entrenamiento completa SIN errores
- [ ] Loss calculado correctamente (no importa el valor)


## PASO 0 — Verificar GPU y configuración

In [ ]:
import torch
import torch.nn as nn
import numpy as np

# Seleccionar GPU si está disponible, si no CPU
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Dispositivo seleccionado: {DEVICE}')

if DEVICE.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memoria GPU disponible: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('⚠️  No hay GPU. Ve a Entorno de ejecución → Cambiar tipo de entorno → GPU T4')

## PASO 1 — Crear imágenes dummy (datos falsos para probar el pipeline)

In [ ]:
import os
from PIL import Image

# ─── Configuración ───────────────────────────────────────────
IMG_SIZE    = 224      # EfficientNet-B0 espera 224×224
N_POR_CLASE = 100      # 100 normales + 100 arritmias = 200 total
BASE_DIR    = 'ecg_dataset_dummy'
SPLITS      = ['train', 'val', 'test']
CLASES      = ['normal', 'arritmia']

# Cuántas imágenes por split (del total N_POR_CLASE)
# train=70%, val=20%, test=10%
SPLIT_SIZES = {'train': 70, 'val': 20, 'test': 10}

# ─── Crear la estructura de carpetas ─────────────────────────
# Estructura esperada por PyTorch ImageFolder:
# ecg_dataset_dummy/
#   train/normal/    train/arritmia/
#   val/normal/      val/arritmia/
#   test/normal/     test/arritmia/

for split in SPLITS:
    for clase in CLASES:
        path = os.path.join(BASE_DIR, split, clase)
        os.makedirs(path, exist_ok=True)

print('Estructura de carpetas creada:')
for split in SPLITS:
    for clase in CLASES:
        path = os.path.join(BASE_DIR, split, clase)
        print(f'  {path}/')

In [ ]:
# ─── Generar imágenes aleatorias ─────────────────────────────
# Cada imagen es ruido aleatorio de 224×224 con 3 canales (RGB)
# No tiene ningún significado médico — solo sirve para probar el código

np.random.seed(42)  # Para reproducibilidad
contador = 0

for split, n_imgs in SPLIT_SIZES.items():
    for clase in CLASES:
        folder = os.path.join(BASE_DIR, split, clase)
        for i in range(n_imgs):
            # Imagen de ruido aleatorio (valores 0-255, 3 canales RGB)
            random_pixels = np.random.randint(0, 255, (IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8)
            img = Image.fromarray(random_pixels, 'RGB')
            img.save(os.path.join(folder, f'ecg_{split}_{clase}_{i:04d}.png'))
            contador += 1

print(f'Total de imágenes creadas: {contador}')
print()

# Verificar cuántas imágenes hay en cada carpeta
for split in SPLITS:
    for clase in CLASES:
        path = os.path.join(BASE_DIR, split, clase)
        n = len(os.listdir(path))
        print(f'  {split}/{clase}: {n} imágenes')

In [ ]:
# ─── Visualizar algunas imágenes dummy ───────────────────────
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 4, figsize=(12, 6))
fig.suptitle('Imágenes Dummy (ruido aleatorio) — solo para probar el pipeline', fontsize=13)

for row, clase in enumerate(CLASES):
    folder = os.path.join(BASE_DIR, 'train', clase)
    files = os.listdir(folder)[:4]
    for col, fname in enumerate(files):
        img = Image.open(os.path.join(folder, fname))
        axes[row, col].imshow(img)
        axes[row, col].set_title(f'{clase} #{col}', fontsize=10)
        axes[row, col].axis('off')

plt.tight_layout()
plt.savefig('dummy_images_preview.png', dpi=80, bbox_inches='tight')
plt.show()
print('Las imágenes reales de Gloria tendrán la misma forma (224×224 PNG) pero con ECGs de verdad.')

## PASO 2 — Implementar el Data Loader con normalización ImageNet

In [ ]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# ─── Transforms ──────────────────────────────────────────────
# Los valores de normalización son los de ImageNet (SIEMPRE usar estos con modelos pre-entrenados)
# mean y std calculados sobre millones de imágenes de ImageNet
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

# Transform para ENTRENAMIENTO (con augmentation básico)
train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),  # Asegurar tamaño correcto
    transforms.ToTensor(),                     # Convierte PIL Image → tensor [0,1]
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)  # Normalizar
])

# Transform para VALIDACIÓN y TEST (sin augmentation — solo normalizar)
val_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
])

# ─── Datasets ────────────────────────────────────────────────
# ImageFolder lee automáticamente las carpetas como clases
# normal/ → clase 0, arritmia/ → clase 1 (orden alfabético)
train_dataset = datasets.ImageFolder(os.path.join(BASE_DIR, 'train'), transform=train_transform)
val_dataset   = datasets.ImageFolder(os.path.join(BASE_DIR, 'val'),   transform=val_transform)
test_dataset  = datasets.ImageFolder(os.path.join(BASE_DIR, 'test'),  transform=val_transform)

print(f'Clases detectadas: {train_dataset.classes}')
print(f'  → clase 0 = {train_dataset.classes[0]}, clase 1 = {train_dataset.classes[1]}')
print()
print(f'Train: {len(train_dataset)} imágenes')
print(f'Val:   {len(val_dataset)} imágenes')
print(f'Test:  {len(test_dataset)} imágenes')

# ─── DataLoaders ─────────────────────────────────────────────
BATCH_SIZE = 32

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f'\nBatch size: {BATCH_SIZE}')
print(f'Batches por epoch (train): {len(train_loader)}')

In [ ]:
# ─── Verificar un batch ──────────────────────────────────────
images, labels = next(iter(train_loader))

print('=== Verificación del Data Loader ===')
print(f'Shape del batch de imágenes: {images.shape}')
print(f'  → (batch_size={images.shape[0]}, canales={images.shape[1]}, alto={images.shape[2]}, ancho={images.shape[3]})')
print(f'Shape del batch de etiquetas: {labels.shape}  → {labels.tolist()[:10]}...')
print(f'Tipo de datos: {images.dtype}')
print(f'Rango de valores (normalizado): [{images.min():.3f}, {images.max():.3f}]')
print()
print('✅ Data loader funciona correctamente.')

## PASO 3 — Implementar el modelo EfficientNet-B0 adaptado

In [ ]:
from torchvision import models

def crear_modelo(device):
    """
    Carga EfficientNet-B0 pre-entrenado en ImageNet
    y reemplaza la última capa para clasificación binaria.
    """
    # Cargar modelo pre-entrenado
    model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)
    
    # Reemplazar la capa final:
    # Original: Linear(1280, 1000) → 1000 clases de ImageNet
    # Nueva:    Linear(1280, 1)    → 1 salida binaria (Normal vs Arritmia)
    model.classifier[1] = nn.Linear(1280, 1)
    
    # Mover el modelo al dispositivo (GPU o CPU)
    model = model.to(device)
    
    return model

# Crear el modelo
model = crear_modelo(DEVICE)

# Verificar parámetros
total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print('=== Modelo EfficientNet-B0 Adaptado ===')
print(f'Total de parámetros:       {total_params:,} ({total_params/1e6:.1f}M)')
print(f'Parámetros entrenables:    {trainable_params:,} ({trainable_params/1e6:.1f}M)')
print(f'Capa final adaptada:       {model.classifier}')
print(f'Modelo en dispositivo:     {next(model.parameters()).device}')

In [ ]:
# ─── Verificar que el modelo acepta nuestro input ────────────
model.eval()
with torch.no_grad():
    dummy_batch = torch.randn(4, 3, 224, 224).to(DEVICE)  # 4 imágenes de prueba
    output = model(dummy_batch)
    probs  = torch.sigmoid(output)  # Convertir logits a probabilidades [0,1]

print('=== Verificación del Forward Pass ===')
print(f'Input shape:   {dummy_batch.shape}')
print(f'Output shape:  {output.shape}  ← logits (sin sigmoid)')
print(f'Output logits: {output.squeeze().tolist()}')
print(f'Probs (sigmoid): {probs.squeeze().tolist()}')
print()
print('✅ El modelo hace forward pass sin errores.')
print('   Probs > 0.5 → Arritmia | Probs < 0.5 → Normal')

## PASO 4 — Configurar el entrenamiento (loss, optimizer)

In [ ]:
# ─── Función de pérdida ──────────────────────────────────────
# BCEWithLogitsLoss = Binary Cross-Entropy + Sigmoid internamente
# Es más estable numéricamente que aplicar sigmoid manualmente y luego BCELoss
criterion = nn.BCEWithLogitsLoss()

# ─── Optimizador ─────────────────────────────────────────────
# Adam es el estándar para transfer learning
# learning_rate = 1e-3 es el valor estándar para la primera fase
LEARNING_RATE = 1e-3
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

print('=== Configuración de Entrenamiento ===')
print(f'Loss:        BCEWithLogitsLoss')
print(f'Optimizer:   Adam')
print(f'LR inicial:  {LEARNING_RATE}')
print(f'Batch size:  {BATCH_SIZE}')
print(f'Dispositivo: {DEVICE}')

## PASO 5 — Ejecutar una epoch completa de entrenamiento

In [ ]:
def train_epoch(model, loader, criterion, optimizer, device):
    """
    Ejecuta una epoch de entrenamiento.
    Retorna: loss promedio y accuracy del epoch.
    """
    model.train()  # Modo entrenamiento (activa BatchNorm y Dropout)
    
    total_loss = 0.0
    correct    = 0
    total      = 0
    
    for batch_idx, (images, labels) in enumerate(loader):
        # Mover datos al dispositivo (GPU/CPU)
        images = images.to(device)
        labels = labels.float().to(device)  # BCELoss necesita float, no long
        
        # ── Forward pass ──
        optimizer.zero_grad()       # Limpiar gradientes del batch anterior
        outputs = model(images)     # Predicciones (logits)
        outputs = outputs.squeeze() # Quitar dimensión extra: (batch,1) → (batch,)
        
        # ── Calcular loss ──
        loss = criterion(outputs, labels)
        
        # ── Backward pass (calcular gradientes) ──
        loss.backward()
        
        # ── Actualizar pesos ──
        optimizer.step()
        
        # ── Acumular métricas ──
        total_loss += loss.item()
        preds = (torch.sigmoid(outputs) > 0.5).float()  # 0 o 1
        correct += (preds == labels).sum().item()
        total   += labels.size(0)
        
        # Progreso
        if (batch_idx + 1) % 2 == 0 or (batch_idx + 1) == len(loader):
            print(f'  Batch {batch_idx+1}/{len(loader)} | Loss: {loss.item():.4f}')
    
    avg_loss = total_loss / len(loader)
    accuracy = correct / total
    return avg_loss, accuracy


def val_epoch(model, loader, criterion, device):
    """
    Evalúa el modelo en el set de validación.
    No actualiza pesos (no hay backward).
    """
    model.eval()  # Modo evaluación (desactiva BatchNorm y Dropout)
    
    total_loss = 0.0
    correct    = 0
    total      = 0
    
    with torch.no_grad():  # No calcular gradientes (ahorra memoria)
        for images, labels in loader:
            images = images.to(device)
            labels = labels.float().to(device)
            
            outputs = model(images).squeeze()
            loss    = criterion(outputs, labels)
            
            total_loss += loss.item()
            preds = (torch.sigmoid(outputs) > 0.5).float()
            correct += (preds == labels).sum().item()
            total   += labels.size(0)
    
    avg_loss = total_loss / len(loader)
    accuracy = correct / total
    return avg_loss, accuracy

In [ ]:
# ─── Ejecutar UNA epoch completa ─────────────────────────────
print('=' * 50)
print('ENTRENANDO EPOCH 1 (datos dummy — solo prueba de pipeline)')
print('=' * 50)

# Entrenamiento
print('\n[TRAIN]')
train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, DEVICE)

# Validación
print('\n[VAL]')
val_loss, val_acc = val_epoch(model, val_loader, criterion, DEVICE)

print()
print('=' * 50)
print('RESULTADO DE EPOCH 1')
print('=' * 50)
print(f'Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} ({train_acc*100:.1f}%)')
print(f'Val   Loss: {val_loss:.4f} | Val   Acc: {val_acc:.4f} ({val_acc*100:.1f}%)')
print()
print('⚠️  Las métricas con datos aleatorios no tienen significado real.')
print('   Lo importante es que el código corrió SIN ERRORES.')
print()
print('✅ PIPELINE LISTO. En Semana 3 solo hay que cambiar BASE_DIR a las imágenes de Gloria.')

## PASO 6 — Resumen visual del pipeline

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Pipeline CNN 2D — ECG Rhythm Analyzer (C2 Datos Dummy)', fontsize=13, fontweight='bold')

# ── Diagrama del flujo de datos ──────────────────────────────
ax = axes[0]
ax.set_xlim(0, 10)
ax.set_ylim(0, 10)
ax.axis('off')
ax.set_title('Flujo de datos', fontsize=11)

steps = [
    (5, 9.0, '📁 Carpetas train/val/test\nnormal/ y arritmia/', '#AED6F1'),
    (5, 7.0, '🔄 ImageFolder + Transforms\nResize(224) → ToTensor → Normalize', '#A9DFBF'),
    (5, 5.0, '📦 DataLoader\nbatch_size=32, shuffle=True', '#F9E79F'),
    (5, 3.0, '🧠 EfficientNet-B0\nLinear(1280→1)', '#F1948A'),
    (5, 1.0, '📉 BCEWithLogitsLoss\n+ Adam optimizer', '#D7BDE2'),
]

for x, y, label, color in steps:
    box = mpatches.FancyBboxPatch((x-3.8, y-0.7), 7.6, 1.4,
                                   boxstyle='round,pad=0.1', facecolor=color,
                                   edgecolor='gray', linewidth=1)
    ax.add_patch(box)
    ax.text(x, y, label, ha='center', va='center', fontsize=8.5)

# Flechas entre pasos
for i in range(len(steps) - 1):
    ax.annotate('', xy=(5, steps[i+1][1]+0.7), xytext=(5, steps[i][1]-0.7),
                arrowprops=dict(arrowstyle='->', color='gray', lw=1.5))

# ── Tabla de configuración ───────────────────────────────────
ax2 = axes[1]
ax2.axis('off')
ax2.set_title('Configuración del experimento', fontsize=11)

tabla = [
    ['Parámetro', 'Valor'],
    ['Modelo',    'EfficientNet-B0'],
    ['Parámetros','5.3M (pre-entrenado ImageNet)'],
    ['Input',     '224×224 px, 3 canales RGB'],
    ['Output',    '1 neurona (logit binario)'],
    ['Loss',      'BCEWithLogitsLoss'],
    ['Optimizer', 'Adam (lr=1e-3)'],
    ['Batch size','32'],
    ['Clases',    '0=normal, 1=arritmia'],
    ['GPU',       str(DEVICE)],
]

table = ax2.table(cellText=tabla[1:], colLabels=tabla[0],
                  loc='center', cellLoc='left')
table.auto_set_font_size(False)
table.set_fontsize(9)
table.scale(1.2, 1.6)

# Header en azul
for j in range(2):
    table[0, j].set_facecolor('#2E86AB')
    table[0, j].set_text_props(color='white', fontweight='bold')

# Filas alternas en gris claro
for i in range(1, len(tabla)):
    color = '#F2F2F2' if i % 2 == 0 else 'white'
    for j in range(2):
        table[i, j].set_facecolor(color)

plt.tight_layout()
plt.savefig('pipeline_c2_resumen.png', dpi=100, bbox_inches='tight')
plt.show()
print('Diagrama guardado como pipeline_c2_resumen.png')

## ✅ Checklist final de C2

In [ ]:
checklist = """
✅ CHECKLIST C2 — DATOS DUMMY
================================

[✓] Estructura de carpetas creada:
    train/normal/, train/arritmia/
    val/normal/,   val/arritmia/
    test/normal/,  test/arritmia/

[✓] 200 imágenes dummy generadas (100 por clase, split 70/20/10)

[✓] EfficientNet-B0 cargado con pesos ImageNet
    → Última capa adaptada: Linear(1280, 1)

[✓] Data loader configurado:
    → ImageFolder leyendo carpetas automáticamente
    → Normalización con valores ImageNet
    → batch_size=32

[✓] Forward pass verificado:
    → Input: (batch, 3, 224, 224)
    → Output: (batch, 1) → probabilidades con sigmoid

[✓] Una epoch completa ejecutada SIN ERRORES:
    → Train loop: forward + backward + optimizer.step()
    → Val loop: evaluación sin gradientes

──────────────────────────────────────
SEMANA 1 COMPLETADA ✓
──────────────────────────────────────

Próximos pasos (Semana 2):
  → C3: Añadir augmentation avanzado, weighted loss, early stopping
  → C4: Implementar transfer learning en dos fases (congelar → descongelar)

Semana 3 (cuando Gloria entregue las imágenes):
  → Cambiar BASE_DIR = 'ecg_dataset_dummy' → ruta a las imágenes reales
  → TODO el código de este notebook sigue funcionando igual
"""
print(checklist)